### Rate Limiting

1 - Semaphore 

- limits how many tasks can run simultaneously
- A semaphore does NOT care about time
-  counter for concurrency

In [1]:
import asyncio

In [2]:
llm_limiter = asyncio.Semaphore(5)
tool_limiter = asyncio.Semaphore(10)
api_limiter = asyncio.Semaphore(20)

In [ ]:
from langchain_openai import ChatOpenAI

In [ ]:
llm = ChatOpenAI(
    model="gpt-4o-mini"
)

LLM Limiter

In [ ]:
async def call_llm(state):

    async with llm_limiter:

        response = await llm.ainvoke(
            state["messages"]
        )

        return {
            "messages": [response]
        }

Tool Limiter

In [ ]:
async def search_tool(query):

    async with tool_limiter:

        print("Calling tool...")

        # Tool logic
        result = f"Search results for {query}"

        return result

API Limiter

In [ ]:
import httpx

In [ ]:
async def call_api(url):

    async with api_limiter:

        async with httpx.AsyncClient() as client:

            response = await client.get(url)

            return response.json()

### 2 -aiolimiter

- limits how many requests happen during a time period.
- like 1000 req per min or 300 req per 30 sec

In [ ]:
from aiolimiter import AsyncLimiter

In [ ]:
llm_limiter = AsyncLimiter(
    10,
    60
)

tool_limiter = AsyncLimiter(
    20,
    60
)

api_limiter = AsyncLimiter(
    30,
    60
)

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

In [ ]:
class State(TypedDict):

    query: str

    llm_response: str

    tool_response: str

    api_response: str

LLM   → 10 requests / minute <br>

Tools → 20 requests / minute<br>

API   → 30 requests / minute

In [ ]:
async def llm_node(state):

    async with llm_limiter:

        response = await llm.ainvoke(
            state["messages"]
        )

        return {
            "messages": [response]
        }

In [ ]:
async def tool_node(state):

    async with tool_limiter:

        result = await search_tool(
            state["query"]
        )

        return {
            "tool_response": result
        }

In [ ]:
async def api_node(state):

    async with api_limiter:

        response = await client.get(
            "https://api.example.com"
        )

        return {
            "api_response": response.json()
        }

In [ ]:
builder = StateGraph(State)


builder.add_node(
    "llm",
    llm_node
)

builder.add_node(
    "tool",
    tool_node
)

builder.add_node(
    "api",
    api_node
)


builder.add_edge(
    START,
    "llm"
)

builder.add_edge(
    "llm",
    "tool"
)

builder.add_edge(
    "tool",
    "api"
)

builder.add_edge(
    "api",
    END
)


graph = builder.compile()

- there can even be limits in seperate api calls

In [ ]:
LIMITERS = {

    "llm":
        AsyncLimiter(10, 60),

    "pubmed":
        AsyncLimiter(20, 60),

    "clinical_trials":
        AsyncLimiter(10, 60),

    "openalex":
        AsyncLimiter(20, 60),

    "tools":
        AsyncLimiter(30, 60)
}

### 3 - Using Fastapi

- many http requests a user can send to your backend API

User A → 10 requests/minute<br>

User B → 20 requests/minute<be>

User C → 5 requests/minute

- pip install slowapi

In [ ]:
from fastapi import FastAPI, Request

from slowapi import Limiter
from slowapi.util import get_remote_address


app = FastAPI()


limiter = Limiter(
    key_func=get_remote_address
)

In [ ]:
@app.post("/chat")
@limiter.limit("10/minute")  # 10 requests per minute per IP address
async def chat(
    request: Request,
    query: str
):

    result = await graph.ainvoke({

        "query": query

    })

    return result

- but it only limit the http request not what that req can entail like 3 api calls ,2 tool calls ,etc